# PathSnap: Exploratory Data Analysis & Spatial Preprocessing
**Course:** UE24CS352A - Machine Learning | PES University  
**Author:** A S Vincent (`vincent2maran@gmail.com`)

This notebook performs:
1. Environment setup and Google Drive mounting (for Google Colab).
2. Inspection of raw files (`.track`, `.nodes`, `.arcs`, `.route`).
3. Spatial coordinate projection (WGS84 degrees -> UTM Zone 32N in metres).
4. Kinematic trajectory diagnostics (speeds, 1 Hz regularity, gaps).
5. High-recall spatial candidate generation using KD-Tree.
6. Interactive Folium map visualization of raw GPS observations vs true road path.


In [2]:
# 1. Google Colab Environment Setup & Dependencies
# Run this cell if running on Google Colab:
!pip install -q pyproj networkx folium scipy scikit-learn

# Mount Google Drive (Optional if data is in Google Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/ML_Hackathon/Student folder/Student folder"

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium

# Local / Colab path detection
LOCAL_DATA = Path("../Student folder/Student folder")
if not LOCAL_DATA.exists():
    LOCAL_DATA = Path("data")
print("Using data directory:", LOCAL_DATA)


'pip' is not recognized as an internal or external command,
operable program or batch file.


Using data directory: data


## 2. Spatial Projection: WGS84 Degrees to UTM Metres
Raw GPS coordinates are in spherical degrees (`lon, lat`). As mandated by the problem guidelines, ordinary Euclidean distance on degrees is invalid because 1 degree of longitude varies with latitude. We project coordinates into **UTM Zone 32N (EPSG:32632)** where unit = 1 metre.


In [3]:
from pyproj import Transformer, CRS

def build_utm_transformer(lon, lat):
    zone = int((lon + 180) / 6) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return Transformer.from_crs(CRS.from_epsg(4326), CRS.from_epsg(epsg), always_xy=False)

def project_coords(lons, lats, transformer):
    eastings, northings = transformer.transform(lats, lons)
    return np.asarray(eastings, dtype=np.float64), np.asarray(northings, dtype=np.float64)

# Test sample coordinates near Darmstadt, Germany
sample_lon, sample_lat = 8.581679, 49.857880
tf = build_utm_transformer(sample_lon, sample_lat)
ex, ny = project_coords([sample_lon], [sample_lat], tf)
print(f"Sample WGS84 ({sample_lon}, {sample_lat}) -> UTM Metric: Easting={ex[0]:.2f}m, Northing={ny[0]:.2f}m")


Sample WGS84 (8.581679, 49.85788) -> UTM Metric: Easting=469931.92m, Northing=5522913.26m


## 3. Load Road Network & Trajectory for Trip 00000001


In [4]:
trip_dir = LOCAL_DATA / "train" / "00000001"
nodes_data = np.loadtxt(trip_dir / "00000001.nodes")
arcs_data = np.loadtxt(trip_dir / "00000001.arcs", dtype=np.int64)
track_data = np.loadtxt(trip_dir / "00000001.track")
route_arcs = np.loadtxt(trip_dir / "00000001.route", dtype=np.int64)

print(f"Road Vertices (Nodes)     : {len(nodes_data):,}")
print(f"Road Segments (Arcs)      : {len(arcs_data):,}")
print(f"GPS Observations (Track)  : {len(track_data):,}")
print(f"Traversed Route Arcs      : {len(route_arcs):,}")


FileNotFoundError: data\train\00000001\00000001.nodes not found.

## 4. Trajectory Sampling Interval Audit (~1 Hz Verification)
We verify that ~97% of observations are spaced 1 second apart, calculate speed, and inspect distributions.


In [ ]:
timestamps = track_data[:, 2]
dt = np.diff(timestamps)

regular_fraction = np.mean(np.abs(dt - 1.0) <= 0.25) * 100
print(f"Percentage of regular 1-second intervals: {regular_fraction:.2f}%")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(dt[dt < 5], bins=30, color='royalblue', edgecolor='black')
plt.title("GPS Time Delta (dt) Distribution")
plt.xlabel("Seconds")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
# Compute speed
dx = np.diff(track_data[:, 0]) * 111320 * np.cos(np.radians(49.85))
dy = np.diff(track_data[:, 1]) * 111320
dist_m = np.hypot(dx, dy)
speed_mps = dist_m / np.maximum(dt, 0.001)

plt.hist(speed_mps[speed_mps < 45], bins=30, color='forestgreen', edgecolor='black')
plt.title("Vehicle Speed Distribution (m/s)")
plt.xlabel("Speed (m/s)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 5. Candidate Generation using KD-Tree
We index road segments and evaluate candidate miss rate against ground truth on Trip 00000001.


In [ ]:
from scipy.spatial import cKDTree

# Build UTM coordinates
tf = build_utm_transformer(np.mean(nodes_data[:, 0]), np.mean(nodes_data[:, 1]))
nodes_x, nodes_y = project_coords(nodes_data[:, 0], nodes_data[:, 1], tf)
track_x, track_y = project_coords(track_data[:, 0], track_data[:, 1], tf)

# Discretize segments into KD-Tree points every 20m
pts, seg_ids = [], []
for arc_id in range(len(arcs_data)):
    fn, tn = arcs_data[arc_id]
    x0, y0 = nodes_x[fn], nodes_y[fn]
    x1, y1 = nodes_x[tn], nodes_y[tn]
    length = np.hypot(x1 - x0, y1 - y0)
    samples = max(2, int(np.ceil(length / 20.0)) + 1)
    for a in np.linspace(0, 1, samples):
        pts.append((x0 + a*(x1 - x0), y0 + a*(y1 - y0)))
        seg_ids.append(arc_id)

tree = cKDTree(np.array(pts))
seg_ids = np.array(seg_ids)

print("KD-Tree built with", len(pts), "sample points!")

# Query radius 60m for first 100 observations
misses = 0
for i in range(100):
    px, py = track_x[i], track_y[i]
    nearby = tree.query_ball_point([px, py], r=60.0)
    cands = set(seg_ids[nearby])
    # Ground truth reference
    true_arc = route_arcs[min(i, len(route_arcs)-1)]
    if true_arc not in cands:
        misses += 1

print(f"Candidate Miss Rate on sample points: {misses / 100 * 100:.2f}% (Target: < 5%)")


## 6. Interactive Folium Map: Raw GPS vs Reconstructed Route
We visualize raw noisy GPS observations (Blue) against the true traversed road segments (Green).


In [ ]:
center_lat = float(track_data[0, 1])
center_lon = float(track_data[0, 0])

fmap = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles="cartodbpositron")

# 1. Plot raw GPS observations (Blue circles)
for i in range(0, min(300, len(track_data)), 2):
    folium.CircleMarker(
        location=[track_data[i, 1], track_data[i, 0]],
        radius=3,
        color="blue",
        fill=True,
        fill_opacity=0.7,
        tooltip=f"GPS Obs {i}"
    ).add_to(fmap)

# 2. Plot ground truth route path (Green polyline)
route_coords = []
for arc_id in route_arcs[:50]:
    fn, tn = arcs_data[arc_id]
    route_coords.append([nodes_data[fn, 1], nodes_data[fn, 0]])
    route_coords.append([nodes_data[tn, 1], nodes_data[tn, 0]])

folium.PolyLine(
    route_coords,
    color="forestgreen",
    weight=4,
    opacity=0.8,
    tooltip="Ground Truth Traversed Road Path"
).add_to(fmap)

fmap.save("folium_trajectory_map.html")
print("Saved interactive map to folium_trajectory_map.html")
fmap
